# Lab 0-04: Tools Give Agents Specific Capabilities

An `LLM` receives text and generates text. To retrieve current information, perform a calculation reliably, create media, access a file, or interact with another service, an agent needs a **tool**: a function that provides one clear capability.

In this notebook, you will explore a small multiplication tool and a weather tool with fixed demonstration data. Sections 1, 3, and 4 run without a model connection. Section 2.1 uses the Qwen model and Ollama endpoint configured in Lab 0-02; it does not require a separate API key.

This notebook adapts the core ideas in the Hugging Face Agents Course lesson, ["What are Tools?"](https://huggingface.co/learn/agents-course/en/unit1/tools).

## What You Will Learn

By the end, you should be able to:

- explain why a tool is different from an `LLM` response
- identify a tool's name, purpose, inputs, outputs, and limits
- follow the basic tool-use loop: the model requests a tool, the agent runs it, and the result becomes new context
- explain why a real weather tool needs an approved source of current information

## 1. What Are AI Tools?

A tool is a function made available to an `LLM` through an agent. It should have one clear objective. The agent can provide many different tools, depending on the task.

| Tool | What it lets an agent do |
| --- | --- |
| Web search | Retrieve current information from the internet. |
| Image generation | Create an image from a text description. |
| Retrieval | Find information in an external source, such as an approved document collection. |
| API interface | Interact with another service, such as GitHub, YouTube, or Spotify. |

These are only examples. A developer can create a tool for any well-defined use case.

A good tool complements the capabilities of an `LLM`. Two common reasons to use one are:

- **Reliable computation:** A calculator tool can produce arithmetic results more reliably than asking the model to calculate in its text response.
- **Current information:** When an answer depends on information that may have changed, an agent can use an approved tool to retrieve it from an appropriate external source, such as a weather provider. The `LLM` then uses the returned result as context rather than guessing.

![Weather tool request and response](figures/weather_tool.png)

*Figure 1. A weather tool receives a location, obtains a weather report, and returns the result for the agent to use as context.*

### 1.1 A Multiplication Tool Example

This multiplication tool illustrates the reliable-computation use case. It has a narrow objective: multiply two whole numbers and return the result.

In [ ]:
# Purpose: define one narrow, reliable capability that an agent can offer to an LLM.
def multiply_integers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

# Call the tool directly once so students can see the result it returns.
multiply_integers(12, 7)

**Components of this tool**

- **Name:** `multiply_integers` clearly states what the tool does.
- **Description:** the docstring, `Multiply two integers.`, explains the tool's objective.
- **Inputs and types:** `a: int` and `b: int` communicate to readers, tools, and the agent that the function expects two integers. These annotations do not by themselves make Python reject other input types.
- **Output type:** `-> int` says that the tool returns one integer: the product of `a` and `b`.

### 1.2 A Weather Tool Example

The following function demonstrates the shape of a weather tool. As Figure 1 shows, in a real agent the tool would send a location to an approved weather provider's API, receive current weather data, and return that result to the agent.

To keep this notebook reliable in every classroom, this version uses fixed demonstration data instead of contacting a live weather service.

In [ ]:
# Purpose: simulate a trusted weather provider without requiring internet access.
DEMO_WEATHER = {
    "Paris": {"condition": "partly cloudy", "temperature_c": 18},
    "New York": {"condition": "light rain", "temperature_c": 16},
}


# This simple annotation says the tool returns a dictionary.
def weather_tool(location: str) -> dict:
    """Return a simulated weather report for a named location."""
    # A real tool would contact an approved provider; this version checks the fixed demo data.
    if location not in DEMO_WEATHER:
        raise ValueError(f"No demonstration weather data for {location}.")
    return {"location": location, **DEMO_WEATHER[location]}

# Run one request to show the kind of structured result a tool can return to an agent.
weather_tool("Paris")

## 2. How Do Tools Work?

**The key issue:** a tool is ordinary code, such as a Python function, but an `LLM` receives text and generates text. The `LLM` cannot directly call a Python function or contact a provider on its own.

The agent program bridges this gap. It describes the available tools to the `LLM`, receives a text-based tool request, checks it, and then runs the approved Python function.

For a user who asks for the product of 12 and 7, the tool-use loop is:

1. **The `LLM` proposes a tool request.** It recognizes that the multiplication tool would help and produces text representing `multiply_integers(12, 7)`.
2. **The agent validates the request.** It checks that the multiplication tool is available and approved.
3. **The agent executes the tool.** It runs the function on the model's behalf and receives the result, `84`.
4. **The result becomes context.** The agent adds `84` to the conversation and asks the `LLM` to write a natural-language response using that result.

Most applications keep these intermediate tool steps out of the user interface. From the user's perspective, the model appears to have calculated the answer itself, but the agent program actually performed the tool call in the background.

In compact form: `user question` → `LLM tool request` → `agent validates and runs tool` → `tool result becomes context` → `LLM response`.

![Tool-use pipeline](figures/tool_pipeline.png)

*Figure 2. The agent passes tool descriptions and the user request to the `LLM`, validates the requested tool, runs approved code, and returns the tool result as new context for the final response.*

### 2.1 How the LLM Produces a Tool Request

The agent puts the tool description and a required request format into the `LLM`'s input. For example, the input can tell the model: `multiply_integers` multiplies two integers, its inputs are `a` and `b`, and a request must use this JSON shape:

```json
{"name": "multiply_integers", "arguments": {"a": 12, "b": 7}}
```

When the user asks, "What is 12 times 7?", the `LLM` predicts that structured text one token at a time. It selects the approved tool name because the request is about multiplication, and it predicts `12` and `7` as the argument values from the user's question. It is choosing a formatted request; it is not running the calculation or calling Python.

The agent receives this JSON-like tool-call output and parses it into the Python dictionary shown in the next cell. In many tool-calling APIs, the provider returns the name and arguments in separate structured fields rather than as literal JSON text. In either case, the agent checks the parsed request before calling any code.

> **How the model chooses a tool:** The agent includes descriptions of the approved tools in the model's input. The `LLM` uses the user's request and those descriptions to predict which tool request best fits—for example, choosing `multiply_integers` for "What is 12 times 7?" This is a prediction, not a guarantee, so the agent still checks the requested tool and arguments before running code.

Section 3 examines tool descriptions and request formats in more detail.

**Important:** The `LLM`'s tool request must follow the agent's tool instructions and schema. It needs an approved tool name and arguments that match the function parameters. Otherwise, the request can fail, so the agent validates it before running any code.

In [ ]:
# json will parse Qwen's tool-request text; Path locates this lab's configuration file.
import json
from pathlib import Path

# These imports read the lab settings and connect to Ollama's OpenAI-compatible endpoint.
from dotenv import dotenv_values
from openai import OpenAI

# Purpose: connect this notebook to the Qwen model configured for Lab 0-04.
# This assumes you opened the notebook from the lab0_04_ai_agent folder.
LAB_NAME = "lab0_04_ai_agent"
lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f"Open this notebook from the {LAB_NAME} folder.")

env_path = lab_dir / ".env"
if not env_path.exists():
    raise FileNotFoundError(
        f"Expected {env_path}. Copy .env.example to .env first."
    )

# Purpose: load the model name and endpoint that the following Qwen demonstration will use.
config = dotenv_values(env_path)
model_name = config.get("MODEL")
ollama_base_url = config.get("OLLAMA_BASE_URL")
if not model_name or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from .env")

# Create the client that sends instructions to Qwen. It does not execute Python tools.
client = OpenAI(base_url=ollama_base_url, api_key="ollama")
print("Model from .env:", model_name)
print("Ollama endpoint:", ollama_base_url)

In [ ]:
# Purpose: show how an agent gives Qwen one tool description and a strict request format.
tool_request_instructions = """
You generate tool requests for an agent.

Available tool:
- Name: multiply_integers
- Description: Multiply two integers.
- Arguments: a (integer), b (integer)

Return exactly one valid JSON object with this structure:
- `name` must be `multiply_integers`.
- `arguments` must contain integer values for `a` and `b`.

Example of valid JSON; choose argument values from the user's question:
{"name": "multiply_integers", "arguments": {"a": 0, "b": 0}}

Do not include Markdown, explanations, or any text outside the JSON object.
""".strip()
# This is the user input from which Qwen should choose the tool and extract arguments.
user_question = "What is 12 times 7?"

# Send the tool instructions as the system message and the task as the user message.
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": tool_request_instructions},
        {"role": "user", "content": user_question},
    ],
    temperature=0,
)

# This is the text Qwen generated; it is not Python code being executed.
raw_tool_request = response.choices[0].message.content
if not raw_tool_request:
    raise ValueError("The model returned no tool-request text. Rerun the cell.")

print("Raw tool-request text from Qwen:\n")
print(raw_tool_request)

# Purpose: convert Qwen's text into data the agent program can inspect before using a tool.
try:
    tool_request = json.loads(raw_tool_request)
except json.JSONDecodeError as error:
    raise ValueError(
        "The model did not return valid JSON. Rerun the cell or check the tool instructions."
    ) from error

print("\nParsed Python dictionary:\n")
print(tool_request)

### 2.2 How the Agent Validates and Uses the Tool

The next cell begins after parsing: `tool_request` is already a Python dictionary. The agent checks that the requested tool is on its approved list, passes the supplied arguments to that function, and receives the result.

> **Note:** To keep the example focused, the code checks only the tool name. It does not validate argument types, permissions, or other limits. A production agent should perform those additional checks before execution.

In [ ]:
# Purpose: demonstrate the agent-side step after Qwen has proposed a tool request.
# `tool_request` is the Python dictionary parsed from Qwen's generated JSON in Section 2.1.
# If you ran only this cell, run the Section 2.1 cells first.

# The agent keeps an allowlist of the tools this workflow permits.
approved_tool_names = {"multiply_integers"}
# Stop if the model requested a tool that is not on the allowlist.
if tool_request["name"] not in approved_tool_names:
    raise ValueError("Requested tool is not approved for this workflow.")

# ** unpacks {"a": 12, "b": 7} into multiply_integers(a=12, b=7).
tool_result = multiply_integers(**tool_request["arguments"])
# In a full agent, this result would be sent back to the LLM as new context.
print("Tool returned:\n")
print(tool_result)

## 3. How Do We Give Tools to an LLM?

The agent usually places descriptions of the available tools in the model's persistent instructions, often called the **system message**. For this to work, each description must be precise about:

- what the tool does
- the exact inputs the tool expects
- the output the tool returns
- important limits on what the tool may do

Tool descriptions are often expressed as structured text, such as JSON or a programming-language signature, because those formats reduce ambiguity. Any consistent, precise format can work.

For a concrete example, `multiply_integers` takes two integers, `a` and `b`, and returns their product as an integer. The next cell creates a description for the `LLM` and inserts it into a system message. The system message tells the model about the tool; it does not let the model execute the Python function directly.

In [ ]:
# Purpose: assemble a manual tool description that an LLM can use when selecting a tool.
# Section 4.1 will show how to generate the tool description automatically.
tools_description = """
Tool Name: multiply_integers
Description: Multiply two integers.
Arguments: a: int, b: int
Output: int
Limits: Expected inputs are two integers; does not retrieve external information.
""".strip()

# Place the tool description into the persistent message that accompanies later user requests.
system_message = f"""
You are an AI assistant designed to provide helpful, precise, and clear responses.

You have access to the following tools:
{tools_description}
""".strip()

print("Tool description sent to the LLM:\n")
print(tools_description)
print("\nSystem message sent to the LLM:\n")
print(system_message)

**Reminder about tool descriptions**

**What the `LLM` learns.** When the agent includes this description in the model's input, the `LLM` can:

- recognize that the tool is available
- identify the required inputs
- anticipate the type of result it will return

**What the agent still does.** The description does not grant access to a tool. The agent validates and executes any requested call.

**Format matters.** There is no universal text format for a tool description. This notebook uses a readable teaching format, while many production frameworks use a structured schema such as JSON Schema. The `LLM`'s tool request often has an even stricter format because the agent must parse and validate it before calling code.

## 4. Making Tool Use More Systematic

When an agent has multiple tools, manual descriptions can become:

- repetitive to write and maintain
- inconsistent across tools
- error-prone when a developer omits or mistypes a detail

These approaches help define, describe, and connect tools in a consistent, repeatable way:

- **Section 4.1:** generate tool descriptions automatically
- **Section 4.2:** wrap tools in a reusable Python structure
- **Section 4.3:** use a standard interface across applications

### 4.1 Generate Tool Descriptions Automatically

Writing every tool description by hand is repetitive and easy to get wrong. A well-written Python function already contains much of the needed information: a meaningful name, a docstring, typed inputs, and an output type. The helper function, `format_tool_description(...)`, inspects those details and turns them into a consistent description.

This is the idea behind many tool libraries: developers write ordinary Python functions, and the library builds a consistent tool description for the `LLM`.

In [ ]:
# Purpose: inspect a Python function so we can generate its tool description automatically.
import inspect

# Purpose: generate an LLM-readable tool description from a Python function.
# The code reads the function's name, docstring, input types, and output type.

# Repeat this small example here so this cell can run independently.
def multiply_integers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


# Convert Python type annotations into readable text for the LLM-facing description.
def type_name(annotation) -> str:
    """Return a readable name for a Python type annotation."""
    return getattr(annotation, "__name__", str(annotation))


# Purpose: build the same tool-description fields from a function instead of writing them by hand.
def format_tool_description(function) -> str:
    """Build an LLM-readable tool description from a Python function."""
    signature = inspect.signature(function)
    arguments = [
        f"{parameter.name}: {type_name(parameter.annotation)}"
        for parameter in signature.parameters.values()
    ]
    description = inspect.getdoc(function) or "No description provided."
    output = type_name(signature.return_annotation)
    return (
        f"Tool Name: {function.__name__}\n"
        f"Description: {description}\n"
        f"Arguments: {', '.join(arguments)}\n"
        f"Output: {output}"
    )

print(format_tool_description(multiply_integers))

### 4.2 Generic Tool Implementation

A generic `Tool` class gives each tool the same interface: a name, a description, inputs, an output, and a callable function. The `@tool` decorator below converts an ordinary typed, documented Python function into a `Tool` object automatically. You do not need to memorize the implementation. Focus on the result: a clear function definition becomes a reusable, consistently described capability.

**How the decorated multiplication tool runs.** After `@tool` is applied, the name `multiply_integers` refers to a `Tool` object that stores the original multiplication function. When code later uses `multiply_integers(12, 7)`, Python automatically invokes the object's `__call__` method. That method passes `12` and `7` to the stored original function and returns `84`. An agent would validate a model's request before making the same call.

This teaching class organizes a tool; it does not enforce permissions or approvals. The surrounding agent workflow must validate a request before it calls the tool.

In [ ]:
# Purpose: define a reusable Tool structure that consistently describes and wraps functions.
import inspect
from collections.abc import Callable

# Repeat this small helper so this cell can run independently.
def type_name(annotation) -> str:
    """Return a readable name for a Python type annotation."""
    return getattr(annotation, "__name__", str(annotation))


# A Tool keeps the function together with the name, description, inputs, and output an LLM needs.
class Tool:
    """A reusable wrapper around one approved Python function."""

    def __init__(self, name: str, description: str, function: Callable, arguments: list[str], output: str):
        self.name = name
        self.description = description
        self.function = function
        self.arguments = arguments
        self.output = output

    # Turn the stored fields into the text an agent can supply to an LLM.
    def to_string(self) -> str:
        """Return the information an LLM needs to know about this tool."""
        return (
            f"Tool Name: {self.name}\n"
            f"Description: {self.description}\n"
            f"Arguments: {', '.join(self.arguments)}\n"
            f"Output: {self.output}"
        )

    def __call__(self, *arguments, **keyword_arguments):
        """Run the wrapped function; the agent must validate the call first."""
        return self.function(*arguments, **keyword_arguments)


# Purpose: make @tool convert an ordinary documented function into a consistent Tool object.
def tool(function: Callable) -> Tool:
    """Turn a typed, documented function into a reusable Tool object."""
    signature = inspect.signature(function)
    arguments = [
        f"{parameter.name}: {type_name(parameter.annotation)}"
        for parameter in signature.parameters.values()
    ]
    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "No description provided.",
        function=function,
        arguments=arguments,
        output=type_name(signature.return_annotation),
    )


# The decorator creates a Tool object; the agent still must approve any later call.
@tool
def multiply_integers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

# Show the generated description that an agent can give to an LLM.
print(multiply_integers.to_string())
# Parentheses trigger Tool.__call__, which delegates to the stored multiplication function.
print(multiply_integers(12, 7))

### 4.3 Model Context Protocol (MCP): A Unified Tool Interface

An individual tool can work inside one application. **Model Context Protocol (MCP)** is an open protocol that standardizes how applications present tools and other context to `LLMs`. You do not need to use MCP in this lab. The big idea is that a standard interface can let different models and agent frameworks use the same approved tools without each application inventing a separate connection method.

MCP can provide:

- a growing list of pre-built integrations that an agent application can make available to its `LLM`
- flexibility to switch between `LLM` providers and vendors
- best practices for securing data within an organization's infrastructure

For forensic work, a standard interface does not remove the need for boundaries: the tool's permissions, approved data scope, logging, and human-review requirements still matter.

## What To Notice

The `LLM` would not receive permission to read every file or make every decision. It receives a precise description of one approved capability. The agent program—not the `LLM`—checks the requested tool and executes it.

A tool can do only what its implementation, agent validation, permissions, and approved scope allow. Its description guides the `LLM`; it does not grant access. This multiplication example is designed for two integer inputs and returns their product; it does not retrieve information or take an external action.

**Try it.** Change `12` or `7` in the tool-request cell. Then explain: What inputs did the tool accept? What was it not able to do?

Next, open [04_tool_selection_demo.ipynb](04_tool_selection_demo.ipynb). You will see Qwen select between addition and multiplication tools before moving to the bounded-agent walkthrough.